# SC4002 Assignment


## Code Initializers

- Import Libraries
- Set SEED to 42
- Use GPU for CUDA programming if have, if not use CPU
- Set max parameters to 20 M


In [17]:
import json
import os
import random
import re
import tarfile
import time
import urllib.request
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('GPU is:', torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device('cpu')
    print('No GPU, use CPU')

DATA_DIR = 'data'
DATA_FILE = os.path.join(DATA_DIR, '1.0', 'data', 'en-US.jsonl')
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
MAX_PARAMS = 20_000_000  

GPU is: NVIDIA GeForce RTX 5060


Loading the dataset and verify via assertions on length of each partition ('train', 'dev' and 'test')

In [18]:
DATASET_URL = 'https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz'

if not os.path.exists(DATA_FILE):
    os.makedirs(DATA_DIR, exist_ok=True)
    archive_path = os.path.join(DATA_DIR, 'massive.tar.gz')
    urllib.request.urlretrieve(DATASET_URL, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extract('1.0/data/en-US.jsonl', DATA_DIR)

records = []
with open(DATA_FILE, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

splits = {'train': [], 'dev': [], 'test': []}
for record in records:
    splits[record['partition']].append(record)

for split_name, examples in splits.items():
    print(f'{split_name:5s}: {len(examples):,} examples')
print('Total:', len(records))

assert len(splits['train']) == 11_514
assert len(splits['dev']) == 2_033
assert len(splits['test']) == 2_974


train: 11,514 examples
dev  : 2,033 examples
test : 2,974 examples
Total: 16521


## Pre-processing

- The field `annot_utt` is used to mark slots as `[type : span]`
- Parsing it via BIO tags
- Verify `annot_utt` is identical to `utt.split()`
- Gold slots sets used for evaluation is directly taken from the parsing of  `(slot type, start, end)`


In [19]:
SLOT_PATTERN = re.compile(r'\[(.*?) : (.*?)\]')

def parse_annotation(annot_utt):
    """Split an annotated utterance into words and their BIO tags."""
    words = []
    tags = []
    position = 0 
    for match in SLOT_PATTERN.finditer(annot_utt):
        for word in annot_utt[position:match.start()].split():
            words.append(word)
            tags.append('O')
        slot_type = match.group(1).strip()
        slot_words = match.group(2).split()
        for i, word in enumerate(slot_words):
            words.append(word)
            if i == 0:
                tags.append('B-' + slot_type)
            else:
                tags.append('I-' + slot_type)
        position = match.end()
    for word in annot_utt[position:].split():
        words.append(word)
        tags.append('O')
    return words, tags


def preprocess(records):
    examples = []
    for record in records:
        words, tags = parse_annotation(record['annot_utt'])
        assert words == record['utt'].split(), f"token mismatch in {record['id']}"
        examples.append({
            'id': record['id'],
            'tokens': [word.lower() for word in words],
            'tags': tags,
            'intent': record['intent'],
        })
    return examples


data = {}
for split_name in ['train', 'dev', 'test']:
    data[split_name] = preprocess(splits[split_name])
intent_labels = sorted({ex['intent'] for ex in data['train']})
slot_types = set()
for ex in data['train']:
    for tag in ex['tags']:
        if tag != 'O':
            slot_types.add(tag[2:]) 
slot_types = sorted(slot_types)
tag_labels = ['O']
for slot_type in slot_types:
    tag_labels.append('B-' + slot_type)
    tag_labels.append('I-' + slot_type)
intent_to_id = {label: i for i, label in enumerate(intent_labels)}
tag_to_id = {label: i for i, label in enumerate(tag_labels)}

PAD = '<pad>'
UNK = '<unk>'
word_counts = Counter()
for ex in data['train']:
    word_counts.update(ex['tokens'])

vocab = [PAD, UNK] + sorted(word_counts)
word_to_id = {word: i for i, word in enumerate(vocab)}

print(f'{len(intent_labels)} intents, {len(slot_types)} slot types, {len(tag_labels)} BIO tags')
print(f'Vocabulary size: {len(vocab):,}')
assert len(intent_labels) == 60
assert len(slot_types) == 55

#Checking for Unknown words in dev and test sets
# for split_name in ['dev', 'test']:
#     total_words = 0
#     unknown_words = 0
#     for ex in data[split_name]:
#         for word in ex['tokens']:
#             total_words += 1
#             if word not in word_to_id:
#                 unknown_words += 1
#     print(f'{split_name}: {100 * unknown_words / total_words:.2f}% of words are unknown')

# example = data['train'][0]
# print('\nTagged example:', list(zip(example['tokens'], example['tags'])), '->', example['intent'])

60 intents, 55 slot types, 111 BIO tags
Vocabulary size: 5,304


## Data packaging for PyTorch 


In [20]:
IGNORE_INDEX = -100  

class MassiveDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, index):
        ex = self.examples[index]
        word_ids = [word_to_id.get(word, word_to_id[UNK]) for word in ex['tokens']]
        tag_ids = [tag_to_id[tag] for tag in ex['tags']]
        intent_id = intent_to_id[ex['intent']]
        return word_ids, tag_ids, intent_id

def collate_batch(batch):
    batch_size = len(batch)
    lengths = torch.tensor([len(word_ids) for word_ids, _, _ in batch])
    max_len = int(lengths.max())
    words = torch.full((batch_size, max_len), word_to_id[PAD], dtype=torch.long)
    tags = torch.full((batch_size, max_len), IGNORE_INDEX, dtype=torch.long)
    intents = torch.zeros(batch_size, dtype=torch.long)
    for i, (word_ids, tag_ids, intent_id) in enumerate(batch):
        words[i, :len(word_ids)] = torch.tensor(word_ids)
        tags[i, :len(tag_ids)] = torch.tensor(tag_ids)
        intents[i] = intent_id
    return words, lengths, tags, intents

def make_loader(split_name, batch_size, shuffle):
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(MassiveDataset(data[split_name]), batch_size=batch_size, shuffle=shuffle,

    collate_fn=collate_batch, generator=generator)

## Evaluation functions for results

* **Intent accuracy** = 100 × correct / N.
* **Slot micro F1**: each prediction is a set of `(slot type, start, end)` spans; a true positive
  needs an exact match of type and span. TP/FP/FN are pooled over all utterances; F1 = 100 × 2TP / (2TP + FP + FN).

In [21]:
def bio_to_spans(tags):
    spans = set()
    current_type = None 
    start = None
    for i, tag in enumerate(list(tags) + ['O']):
        continues_current_slot = tag.startswith('I-') and tag[2:] == current_type
        if continues_current_slot:
            continue
        if current_type is not None:
            spans.add((current_type, start, i))
        if tag == 'O':
            current_type, start = None, None
        else:
            current_type, start = tag[2:], i
    return spans

def intent_accuracy(gold_intents, predicted_intents):
    correct = 0
    for gold, predicted in zip(gold_intents, predicted_intents):
        if gold == predicted:
            correct += 1
    return 100.0 * correct / len(gold_intents)

def slot_micro_f1(gold_spans, predicted_spans):
    true_positive = 0
    false_positive = 0
    false_negative = 0
    for gold, predicted in zip(gold_spans, predicted_spans):
        true_positive += len(gold & predicted)
        false_positive += len(predicted - gold)
        false_negative += len(gold - predicted)
    if true_positive + false_positive + false_negative == 0:
        return 100.0
    return 100.0 * 2 * true_positive / (2 * true_positive + false_positive + false_negative)

def evaluate_predictions(examples, predicted_intents, predicted_tags):
    gold_intents = [ex['intent'] for ex in examples]
    gold_spans = [bio_to_spans(ex['tags']) for ex in examples]
    predicted_spans = [bio_to_spans(tags) for tags in predicted_tags]
    accuracy = intent_accuracy(gold_intents, predicted_intents)
    f1 = slot_micro_f1(gold_spans, predicted_spans)
    return {'intent_acc': accuracy, 'slot_f1': f1, 'combined': (accuracy + f1) / 2}

assert bio_to_spans(['O', 'B-time', 'I-time', 'B-date']) == {('time', 1, 3), ('date', 3, 4)}
assert bio_to_spans(['I-time', 'I-date']) == {('time', 0, 1), ('date', 1, 2)}

perfect_intents = [ex['intent'] for ex in data['dev']]
perfect_tags = [ex['tags'] for ex in data['dev']]
perfect_scores = evaluate_predictions(data['dev'], perfect_intents, perfect_tags)
assert perfect_scores == {'intent_acc': 100.0, 'slot_f1': 100.0, 'combined': 100.0}


## Functions for: Training only using train data. Model seclection with dev and Final evaluation with 'test'.

- Logging epochs showing train loss, dev loss, dev intent accuracy, dev slot F1, and elapsed time
- Save highest combined score -> (intent accuracy + slot micro F1) / 2


In [22]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad() 
def predict(model, split_name, batch_size=256):
    model.eval()
    total_loss = 0.0
    num_examples = 0
    predicted_intents = []
    predicted_tags = []

    for words, lengths, tags, intents in make_loader(split_name, batch_size, shuffle=False):
        words = words.to(DEVICE)
        tags = tags.to(DEVICE)
        intents = intents.to(DEVICE)
        intent_scores, slot_scores = model(words, lengths)
        loss = model.loss(intent_scores, slot_scores, intents, tags)
        total_loss += loss.item() * len(words)
        num_examples += len(words)
        for intent_id in intent_scores.argmax(dim=-1).tolist():
            predicted_intents.append(intent_labels[intent_id])
        tag_ids_per_sentence = model.decode_slots(slot_scores, lengths)
        for tag_ids, length in zip(tag_ids_per_sentence, lengths.tolist()):
            predicted_tags.append([tag_labels[t] for t in tag_ids[:length]])
    return total_loss / num_examples, predicted_intents, predicted_tags

def evaluate(model, split_name):
    loss, predicted_intents, predicted_tags = predict(model, split_name)
    scores = evaluate_predictions(data[split_name], predicted_intents, predicted_tags)
    scores['loss'] = loss
    return scores

def train_model(model, name, epochs, lr, batch_size, weight_decay=0.0, clip=5.0):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'{name}_best.pt')
    history = []
    best = {'combined': -1}
    training_start = time.time()
    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        num_examples = 0
        for words, lengths, tags, intents in make_loader('train', batch_size, shuffle=True):
            words = words.to(DEVICE)
            tags = tags.to(DEVICE)
            intents = intents.to(DEVICE)
            intent_scores, slot_scores = model(words, lengths)
            loss = model.loss(intent_scores, slot_scores, intents, tags)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)  # stops very large, unstable updates
            optimizer.step()
            total_loss += loss.item() * len(words)
            num_examples += len(words)

        dev = evaluate(model, 'dev')
        epoch_time = time.time() - epoch_start
        total_time = time.time() - training_start
        history.append({
            'epoch': epoch,
            'train_loss': total_loss / num_examples,
            'dev_loss': dev['loss'],
            'dev_intent_acc': dev['intent_acc'],
            'dev_slot_f1': dev['slot_f1'],
            'dev_combined': dev['combined'],
            'epoch_time_s': epoch_time,
            'elapsed_s': total_time,
        })

        note = ''
        if dev['combined'] > best['combined']:
            best = {'epoch': epoch, **dev}
            torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'dev_metrics': dev}, checkpoint_path)
            note = '  <- best, saved'

        print(f"epoch {epoch:2d} | train loss {total_loss / num_examples:.4f} | dev loss {dev['loss']:.4f} | "
              f"dev intent acc {dev['intent_acc']:.2f} | dev slot F1 {dev['slot_f1']:.2f} | "
              f"combined {dev['combined']:.2f} | {epoch_time:.1f}s (total {total_time:.1f}s){note}")

    print(f"\nBest checkpoint: epoch {best['epoch']} | dev intent acc {best['intent_acc']:.2f} | "
          f"dev slot F1 {best['slot_f1']:.2f} | dev combined {best['combined']:.2f} | file {checkpoint_path}")
    return history, best, checkpoint_path


def final_evaluation(model, checkpoint_path, best):
    """Load the saved best model and score it on dev and test, with no further training."""
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.to(DEVICE)
    print(f"Loaded {checkpoint_path} (epoch {checkpoint['epoch']})")
    results = {}
    for split_name in ['dev', 'test']:
        scores = evaluate(model, split_name)
        results[split_name] = scores
        print(f"{split_name:5s} | intent accuracy {scores['intent_acc']:.2f}% | "
              f"slot micro F1 {scores['slot_f1']:.2f}% | combined {scores['combined']:.2f}")
    assert abs(results['dev']['combined'] - best['combined']) < 1e-6, 'dev score differs from training run'
    print('Reloaded checkpoint reproduces the best validation score.')
    return results

# One direction LSTM tagger + classifier

- Word embedding (dim 100, randomly initialised, trained from scratch).
- One shared single-layer **left-to-right LSTM** encoder (hidden 128).
- **Slot head:** linear layer on each LSTM output → 111 BIO tags (argmax per token).
- **Intent head:** final LSTM hidden state (after reading the last word) → linear layer → 60 intents.
- **Outputs.** One intent label and a BIO sequence per utterance, decoded into `[slot type : span]` pairs.
- **Loss.** Joint loss = cross-entropy(intent) + token-level cross-entropy(slots) (padding ignored).
- **Optimiser.** Adam, lr 1e-3, batch 32, gradient clipping at 5, 15 epochs, no regularisation / dropout.

In [23]:
class JointLSTM(nn.Module):
    def __init__(self, vocab_size, num_intents, num_tags, embedding_dim=100, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True) #one direction left to right
        self.slot_head = nn.Linear(hidden_size, num_tags) 
        self.intent_head = nn.Linear(hidden_size, num_intents) 
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def forward(self, words, lengths):
        embedded = self.embedding(words)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (final_hidden, _) = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=words.size(1))
        slot_scores = self.slot_head(outputs)  
        sentence_summary = final_hidden[-1]   
        intent_scores = self.intent_head(sentence_summary)
        return intent_scores, slot_scores

    def loss(self, intent_scores, slot_scores, intents, tags):
        intent_loss = self.loss_fn(intent_scores, intents)
        slot_loss = self.loss_fn(slot_scores.reshape(-1, slot_scores.size(-1)), tags.reshape(-1))
        return intent_loss + slot_loss

    def decode_slots(self, slot_scores, lengths):
        return slot_scores.argmax(dim=-1).tolist()


set_seed()
baseline = JointLSTM(len(vocab), len(intent_labels), len(tag_labels))
print(baseline)

total_params, trainable_params = count_params(baseline)
print(f'Total parameters: {total_params:,} | trainable: {trainable_params:,}')
assert total_params <= MAX_PARAMS, 'model exceeds the 20M parameter cap'

JointLSTM(
  (embedding): Embedding(5304, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (slot_head): Linear(in_features=128, out_features=111, bias=True)
  (intent_head): Linear(in_features=128, out_features=60, bias=True)
  (loss_fn): CrossEntropyLoss()
)
Total parameters: 670,219 | trainable: 670,219


## Actual Training and Selection on train and dev data respectively

In [24]:
set_seed()
BASELINE_EPOCHS = 15

baseline_history, baseline_best, baseline_checkpoint = train_model(
    baseline,
    name='stage1_baseline',
    epochs=BASELINE_EPOCHS,
    lr=1e-3,
    batch_size=32,
)

epoch  1 | train loss 4.6934 | dev loss 3.2284 | dev intent acc 47.86 | dev slot F1 18.62 | combined 33.24 | 3.3s (total 3.3s)  <- best, saved
epoch  2 | train loss 2.5651 | dev loss 2.1575 | dev intent acc 66.70 | dev slot F1 33.60 | combined 50.15 | 3.4s (total 6.6s)  <- best, saved
epoch  3 | train loss 1.7627 | dev loss 1.7326 | dev intent acc 73.14 | dev slot F1 41.79 | combined 57.47 | 3.6s (total 10.3s)  <- best, saved
epoch  4 | train loss 1.3356 | dev loss 1.5136 | dev intent acc 76.93 | dev slot F1 47.57 | combined 62.25 | 4.1s (total 14.4s)  <- best, saved
epoch  5 | train loss 1.0556 | dev loss 1.4061 | dev intent acc 77.42 | dev slot F1 52.46 | combined 64.94 | 3.7s (total 18.1s)  <- best, saved
epoch  6 | train loss 0.8510 | dev loss 1.3349 | dev intent acc 78.95 | dev slot F1 55.24 | combined 67.09 | 3.1s (total 21.2s)  <- best, saved
epoch  7 | train loss 0.6959 | dev loss 1.2879 | dev intent acc 79.59 | dev slot F1 57.62 | combined 68.60 | 3.3s (total 24.5s)  <- best, 

**What this cell does:** plots the learning curves. Left: training vs dev loss per epoch — when dev loss
starts rising while training loss keeps falling, the model is starting to overfit (memorise). Right: dev
intent accuracy, slot F1 and combined score per epoch.

## Final evaluation

In [25]:
baseline_for_eval = JointLSTM(len(vocab), len(intent_labels), len(tag_labels))
baseline_results = final_evaluation(baseline_for_eval, baseline_checkpoint, baseline_best)

Loaded checkpoints\stage1_baseline_best.pt (epoch 14)
dev   | intent accuracy 79.39% | slot micro F1 63.50% | combined 71.45
test  | intent accuracy 78.68% | slot micro F1 57.87% | combined 68.28
Reloaded checkpoint reproduces the best validation score.
